# Explainable Malware Detection — Training (Colab)

Loads EMBER-2018 and BODMAS from Google Drive (both already uploaded,
under `5th Sem - Nmims/Dataset`), trains the clustering-aided LightGBM
model, evaluates it on EMBER, then on BODMAS with no retraining.

## 1. Mount Drive, clone the repo, install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/5th Sem - Nmims/Dataset"
DRIVE_CACHE = DRIVE_DATA + "/ember_vectorized"
DRIVE_MODELS = "/content/drive/MyDrive/5th Sem - Nmims/explainable-malware-detection/models"

!git clone https://github.com/Rishi-V1/Bubo-AI-.git repo
%cd repo/explainable-malware-detection

!pip install -q pandas numpy scikit-learn lightgbm shap lime alibi fastapi uvicorn streamlit joblib
!pip install -q lief
!pip install -q git+https://github.com/elastic/ember.git

### 1.1 Patch a known EMBER/scikit-learn incompatibility

EMBER's `features.py` calls `FeatureHasher.transform()` with a plain
string where newer scikit-learn requires a list of strings. Confirmed bug,
not a local setup issue: https://github.com/elastic/ember/issues/103.
This patches the installed package every run, since a fresh Colab runtime
reinstalls it from scratch each time.

In [ ]:
import ember as _ember
import os

features_path = os.path.join(os.path.dirname(_ember.__file__), "features.py")
!sed -i "s/transform(\[raw_obj\['entry'\]\])/transform([[raw_obj['entry']]])/" "{features_path}"
!grep -n "transform(\[\[raw_obj" "{features_path}" 

## 2. EMBER-2018

First run: extracts the archive from Drive and vectorizes it, then saves
the vectorized `.dat` files back to Drive. Every run after that just
copies the already-vectorized files — skips extraction and vectorization
entirely, which is the slow part.

In [ ]:
import os, shutil

EMBER_DIR = "/content/ember2018"
os.makedirs(EMBER_DIR, exist_ok=True)

if os.path.exists(os.path.join(DRIVE_CACHE, "X_train.dat")):
    for fname in os.listdir(DRIVE_CACHE):
        shutil.copy(os.path.join(DRIVE_CACHE, fname), EMBER_DIR)
else:
    archive = os.path.join(DRIVE_DATA, "ember_dataset_2018_2.tar.bz2")
    !tar -xjf "{archive}" -C /content/
    !mv /content/ember2018/* {EMBER_DIR}/ 2>/dev/null || true

    import ember
    ember.create_vectorized_features(EMBER_DIR, feature_version=2)

    os.makedirs(DRIVE_CACHE, exist_ok=True)
    for fname in os.listdir(EMBER_DIR):
        if fname.endswith(".dat"):
            shutil.copy(os.path.join(EMBER_DIR, fname), DRIVE_CACHE)

os.listdir(EMBER_DIR)

## 3. BODMAS

Searches the `Bodmas` folder on Drive for the `.npz` file, however deep
it's nested, so it works whether it's directly inside `Bodmas/` or one
level deeper in `Bodmas/bodmas/`.

In [ ]:
import glob
import numpy as np

bodmas_matches = glob.glob(os.path.join(DRIVE_DATA, "Bodmas", "**", "*.npz"), recursive=True)
assert bodmas_matches, "No .npz file found under the Bodmas folder on Drive."

bodmas_npz_path = bodmas_matches[0]
print(f"Using: {bodmas_npz_path}")

data = np.load(bodmas_npz_path)
X_bodmas_raw, y_bodmas = data["X"], data["y"]
print(f"BODMAS loaded: X={X_bodmas_raw.shape}, malware ratio={y_bodmas.mean():.3f}")

## 4. Load EMBER using the repo's own loader

In [ ]:
import sys
sys.path.insert(0, "/content/repo/explainable-malware-detection")

from src.features.load_dataset import load_ember_subset

X_train, y_train = load_ember_subset(EMBER_DIR, n_samples=None, split="train")
X_test, y_test = load_ember_subset(EMBER_DIR, n_samples=None, split="test")

print(f"Train: {X_train.shape}, malware ratio {y_train.mean():.3f}")
print(f"Test:  {X_test.shape}, malware ratio {y_test.mean():.3f}")

## 5. Train the clustering-aided LightGBM model

K-means groups the training data into clusters; each cluster gets its own
LightGBM model (Dener & Gulburun, 2023), adapted to LightGBM only so
TreeSHAP stays exact across every cluster.

In [ ]:
from src.models.clustered_lightgbm import ClusteredLightGBM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

model = ClusteredLightGBM(n_clusters=3)

t0 = time.time()
model.fit(X_train.values, y_train.values)
print(f"Training took {time.time()-t0:.1f}s")

y_pred = model.predict(X_test.values)
y_proba = model.predict_proba(X_test.values)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print()
print("Base paper (Al Siam et al., 2026) on EMBER:")
print("Accuracy 0.9116, Precision 0.8852, Recall 0.9422, F1 0.9128, ROC-AUC 0.974")

### 5.1 Cluster count sweep

In [ ]:
results = {}
for k in [2, 3, 4, 5]:
    m = ClusteredLightGBM(n_clusters=k)
    m.fit(X_train.values, y_train.values)
    acc = accuracy_score(y_test, m.predict(X_test.values))
    results[k] = acc
    print(f"n_clusters={k}: accuracy={acc:.4f}")

best_k = max(results, key=results.get)
print(f"\nBest: n_clusters={best_k}, accuracy={results[best_k]:.4f}")

## 6. Flat LightGBM baseline

In [ ]:
import lightgbm as lgb
from src.evaluation.cross_dataset import evaluate

flat_model = lgb.LGBMClassifier(
    objective="binary", n_estimators=300, num_leaves=64,
    learning_rate=0.05, verbosity=-1, random_state=42,
)
flat_model.fit(X_train.values, y_train.values)

flat_metrics = evaluate(flat_model, X_test.values, y_test.values)
flat_metrics

## 7. Cross-dataset evaluation: EMBER to BODMAS

In [ ]:
import pandas as pd
from src.evaluation.cross_dataset import full_comparison_report

X_bodmas = pd.DataFrame(X_bodmas_raw)

# The actual novelty experiment (docs/PROJECT_STATUS.md Section 4): does
# clustering's advantage over the flat baseline, if any, survive the
# EMBER -> BODMAS shift, or does it evaporate along with (or faster than)
# the flat model's own accuracy?
report = full_comparison_report(
    clustered_model=model,
    flat_model=flat_model,
    X_source_test=X_test.values, y_source_test=y_test.values,
    X_target=X_bodmas.values, y_target=y_bodmas,
    source_name="EMBER", target_name="BODMAS",
)

import json
print(json.dumps(report, indent=2))

### 7.1 RQ2: do explanations stay fidelity-consistent under the shift?

Per Mohseni et al. (Paper 2, docs/PROJECT_STATUS.md Section 3), SHAP
specifically is predicted to become unreliable under distribution shift.
This runs the same fidelity/consistency check on a sample from each
dataset and compares them -- a real drop from EMBER to BODMAS (not just a
different number) is the actual finding, not either number in isolation.

Uses a small sample (fidelity/consistency are both per-instance and not
cheap -- SHAP's TreeExplainer and especially LIME's perturbation sampling
add up over thousands of rows). A few hundred samples is enough to see
whether the two distributions of scores clearly differ.

In [ ]:
import numpy as np
from src.explainability import ShapExplainer, LimeExplainer
from src.evaluation.explanation_stability import shap_fidelity, lime_fidelity

STABILITY_SAMPLE_SIZE = 200
rng = np.random.default_rng(42)

ember_idx = rng.choice(len(X_test), size=min(STABILITY_SAMPLE_SIZE, len(X_test)), replace=False)
bodmas_idx = rng.choice(len(X_bodmas), size=min(STABILITY_SAMPLE_SIZE, len(X_bodmas)), replace=False)

shap_explainer = ShapExplainer(model)
lime_explainer = LimeExplainer(model, X_background=X_train.values[:200])

ember_shap_fidelity = shap_fidelity(shap_explainer, model, X_test.values[ember_idx])
bodmas_shap_fidelity = shap_fidelity(shap_explainer, model, X_bodmas.values[bodmas_idx])

ember_lime_fidelity = lime_fidelity(lime_explainer, X_test.values[ember_idx]).mean()
bodmas_lime_fidelity = lime_fidelity(lime_explainer, X_bodmas.values[bodmas_idx]).mean()

print(f"SHAP fidelity  -- EMBER: {ember_shap_fidelity:.4f}   BODMAS: {bodmas_shap_fidelity:.4f}")
print(f"LIME fidelity  -- EMBER: {ember_lime_fidelity:.4f}   BODMAS: {bodmas_lime_fidelity:.4f}")
print()
print("Base paper (Al Siam et al., 2026) on EMBER: LIME 0.86, SHAP 0.91 (fidelity)")

## 8. Save the trained model

In [ ]:
import joblib

os.makedirs(DRIVE_MODELS, exist_ok=True)
joblib.dump(model, os.path.join(DRIVE_MODELS, "clustered_lightgbm_ember.joblib"))
print(f"Saved to {DRIVE_MODELS}")

### 8.1 Save a background sample for the explainability backend

LIME and Anchors (`src/explainability/`) need a small real sample of
training data at runtime -- not the full dataset, just enough to learn
feature statistics / build a discretizer. Doesn't require retraining,
just run this right after section 8 while `X_train` is still in memory.

In [ ]:
import numpy as np

BACKGROUND_SAMPLE_SIZE = 200
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=min(BACKGROUND_SAMPLE_SIZE, len(X_train)), replace=False)
background_sample = X_train.values[idx]

np.save(os.path.join(DRIVE_MODELS, "background_sample.npy"), background_sample)
print(f"Saved background sample {background_sample.shape} to {DRIVE_MODELS}")

## 9. Phase 5: build the human-eval survey stimulus set (RQ3)

Builds the blinded EMBER + BODMAS stimulus set the analyst trust survey
presents to participants (`src/evaluation/survey_design.py`). Needs
`model`, `X_test`/`y_test` (EMBER), and `X_bodmas`/`y_bodmas` (section 7)
still in memory -- run this in the same session as sections 4 and 7, not
a fresh restart.

Two blinding decisions matter here (see survey_design.py's docstring for
the full rationale): method identity (SHAP/LIME/Anchors) is relabeled
"A/B/C" per stimulus, reshuffled every time, and dataset identity
(EMBER/BODMAS) never appears in what participants see -- both are kept
only in the separate answer key, joined back in by `survey_analysis.py`
once responses come in.

`include_anchors=True` here (unlike the sandbox demo run in
`data/survey/`, which skipped Anchors entirely -- see
`docs/phase5_survey_methodology.md`) because Colab's runtime has enough
RAM headroom for `anchors_explainer.py`'s reduced
`batch_size=50, binary_cache_size=500` defaults to not OOM the way the
tighter sandbox container did. If this cell OOMs or runs for a very long
time anyway (Anchors' search can simply be slow at EMBER's
dimensionality, independent of memory -- see `engine.py`'s reliability
note), set `include_anchors=False` and re-run; every stimulus still gets
all three labels, with Anchors' card replaced by a clearly marked
placeholder, so the survey's 3-way design stays intact either way.

In [ ]:
import sys
sys.path.insert(0, "/content/repo/explainable-malware-detection")

from src.explainability import ExplanationEngine
from src.evaluation.survey_design import build_stimulus_set

SURVEY_DIR = os.path.join(DRIVE_MODELS, "..", "survey")
os.makedirs(SURVEY_DIR, exist_ok=True)

feature_names = [f"f{i}" for i in range(X_train.shape[1])]
engine = ExplanationEngine(model, X_background=X_train.values[:200], feature_names=feature_names)

stimulus_set = build_stimulus_set(
    engine,
    datasets={
        "EMBER": (X_test.values, y_test.values),
        "BODMAS": (X_bodmas.values, y_bodmas),
    },
    n_per_dataset=10,
    top_k=5,
    include_anchors=True,   # flip to False if this OOMs or takes too long -- see note above
)

stimuli_path = os.path.join(SURVEY_DIR, "stimuli.json")
answer_key_path = os.path.join(SURVEY_DIR, "answer_key.json")
stimulus_set.save(stimuli_path, answer_key_path)

print(f"Built {len(stimulus_set.stimuli)} stimuli ({10} per dataset x 2 datasets)")
print(f"Participant-facing file : {stimuli_path}")
print(f"Answer key (keep private): {answer_key_path}")
print()
print("Next: paste the contents of stimuli.json into the STIMULI constant in the")
print("survey artifact (or re-publish it with Claude Code) to replace the sandbox")
print("demo set, then keep answer_key.json for survey_analysis.py once responses")
print("come in.")